In [1]:
!pip install evaluate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.6 MB/s eta 0:00:00


In [2]:
import torch.nn as nn
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset, concatenate_datasets
import numpy as np
from torch.utils.data import DataLoader, Dataset

device = "cuda" if torch.cuda.is_available() else "cpu"

# Loading Model

In [41]:
model = AutoModelForCausalLM.from_pretrained(
    "Pankaj121212/PM_MiniFinLLM", trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(
    "Pankaj121212/PM_MiniFinLLM", trust_remote_code=True
)

modeling_PM_MiniFinLLM.py: 0.00B [00:00, ?B/s]

arch.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Pankaj121212/PM_MiniFinLLM:
- arch.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/Pankaj121212/PM_MiniFinLLM:
- modeling_PM_MiniFinLLM.py
- arch.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

PM_MiniFinLLM_Model LOAD REPORT from: Pankaj121212/PM_MiniFinLLM
Key                 | Status  | 
--------------------+---------+-
model.linear.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

In [4]:
print(list(model.state_dict().keys())[:5])

['model.embedding.weight', 'model.llm_layers.0.linear1.weight', 'model.llm_layers.0.linear1.bias', 'model.llm_layers.0.lyr_norm1.weight', 'model.llm_layers.0.lyr_norm1.bias']


In [5]:
print(model.state_dict()["model.embedding.weight"][0][:5])

tensor([-0.1794, -0.4016, -0.3976, -0.5725,  0.2631])


In [6]:
print(model.state_dict()["model.linear.weight"][0][:5])

tensor([-0.1794, -0.4016, -0.3976, -0.5725,  0.2631])


# Loading Dataset

In [5]:
# because model is small so i will be using a mix of virattt/financial-qa-10K ,sweatSmile/FinanceQA and LLukas22/fiqa
# not using any multi turn dataset because of model size it is outside its capacity

In [ ]:
!hf auth login --token your_hf_token

A new version of huggingface_hub (1.16.1) is available! You are using version 1.4.1.
To update, run: pip install -U huggingface_hub

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: write).
The token `MyLLm` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `MyLLm`


In [7]:
context_dataset1 = load_dataset("virattt/financial-qa-10K")
context_dataset2 = load_dataset("sweatSmile/FinanceQA")
qa_dataset = load_dataset("LLukas22/fiqa")

README.md:   0%|          | 0.00/419 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.59M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/441k [00:00<?, ?B/s]

...

README.md: 0.00B [00:00, ?B/s]

train.json:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

test.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14511 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2561 [00:00<?, ? examples/s]

In [8]:
context_dataset1, context_dataset2

(DatasetDict({
     train: Dataset({
         features: ['question', 'answer', 'context', 'ticker', 'filing'],
         num_rows: 7000
     })
 }),
 DatasetDict({
     train: Dataset({
         features: ['COMPANY_ID', 'QUERY', 'ANSWER', 'CONTEXT', '__index_level_0__'],
         num_rows: 3705
     })
     test: Dataset({
         features: ['COMPANY_ID', 'QUERY', 'ANSWER', 'CONTEXT', '__index_level_0__'],
         num_rows: 927
     })
 }))

In [9]:
qa_dataset

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 14511
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 2561
    })
})

In [10]:
def prerpocess_context_dataset1(row):
    question = row["question"]
    answer = row["answer"]
    context = row["context"]
    inp = f"Answer the following question based on the given context \nQuestion - {question} \nContext - {context}.\n"
    return {
        "messages": [
            {"role": "user", "content": inp},
            {"role": "assistant", "content": answer},
        ]
    }

In [11]:
def prerpocess_context_dataset2(row):
    question = row["QUERY"]
    answer = row["ANSWER"]
    context = row["CONTEXT"]
    inp = f"Answer the following question based on the given context \nQuestion - {question} \nContext - {context}.\n"
    return {
        "messages": [
            {"role": "user", "content": inp},
            {"role": "assistant", "content": answer},
        ]
    }

In [12]:
def preprocess_qa_dataset(row):
    question = row["question"]
    answer = row["answer"]
    inp = question
    return {
        "messages": [
            {"role": "user", "content": inp},
            {"role": "assistant", "content": answer},
        ]
    }

In [13]:
print(preprocess_qa_dataset(qa_dataset["train"][0])["messages"])

[{'role': 'user', 'content': 'What do brokers do with bad stock?'}, {'role': 'assistant', 'content': "For every seller, there's a buyer. Buyers may have any reason for wanting to buy (bargain shopping, foolish belief in a crazy business, etc).  The party (brokerage, market maker, individual) owning the stock at the time the company goes out of business is the loser . But in a general panic, not every company is going to go out of business. So the party owning those stocks can expect to recover some, or all, of the value at some point in the future. Brokerages all reserve the right to limit margin trading (required for short selling), and during a panic would likely not allow you to short a stock they feel is a high risk for them."}]


In [14]:
print(prerpocess_context_dataset1(context_dataset1["train"][0])["messages"])

[{'role': 'user', 'content': 'Answer the following question based on the given context \nQuestion - What area did NVIDIA initially focus on before expanding to other computationally intensive fields? \nContext - Since our original focus on PC graphics, we have expanded to several other large and important computationally intensive fields..\n'}, {'role': 'assistant', 'content': 'NVIDIA initially focused on PC graphics.'}]


In [15]:
print(prerpocess_context_dataset2(context_dataset2["train"][0])["messages"])

[{'role': 'user', 'content': "Answer the following question based on the given context \nQuestion - What is the equity share capital of the company? \nContext - Symbol: ARCOTECH Company Name: Arcotech Ltd. EQUITIES AND LIABILITIES: nan SHAREHOLDER'S FUNDS: nan Equity Share Capital: 21 Total Share Capital: 24.32 Reserves and Surplus: -113.53 Total Reserves and Surplus: -113.53 Total Shareholders Funds: -89.21 NON-CURRENT LIABILITIES: nan Long Term Borrowings: 0 Deferred Tax Liabilities [Net]: 0 Other Long Term.\n"}, {'role': 'assistant', 'content': 'The equity share capital of the company is 21.'}]


In [16]:
qa_dataset_preprocessed = qa_dataset.map(preprocess_qa_dataset)
context_dataset_preprocessed1 = context_dataset1.map(prerpocess_context_dataset1)
context_dataset_preprocessed2 = context_dataset2.map(prerpocess_context_dataset2)

Map:   0%|          | 0/14511 [00:00<?, ? examples/s]

Map:   0%|          | 0/2561 [00:00<?, ? examples/s]

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3705 [00:00<?, ? examples/s]

Map:   0%|          | 0/927 [00:00<?, ? examples/s]

In [17]:
qa_dataset_preprocessed = qa_dataset_preprocessed.remove_columns(["question", "answer"])

In [18]:
context_dataset_preprocessed1 = context_dataset_preprocessed1.remove_columns(
    ["question", "answer", "context", "ticker", "filing"]
)
context_dataset_preprocessed2 = context_dataset_preprocessed2.remove_columns(
    ["COMPANY_ID", "QUERY", "ANSWER", "CONTEXT", "__index_level_0__"]
)

In [19]:
qa_dataset_preprocessed, context_dataset_preprocessed1, context_dataset_preprocessed2

(DatasetDict({
     train: Dataset({
         features: ['messages'],
         num_rows: 14511
     })
     test: Dataset({
         features: ['messages'],
         num_rows: 2561
     })
 }),
 DatasetDict({
     train: Dataset({
         features: ['messages'],
         num_rows: 7000
     })
 }),
 DatasetDict({
     train: Dataset({
         features: ['messages'],
         num_rows: 3705
     })
     test: Dataset({
         features: ['messages'],
         num_rows: 927
     })
 }))

In [20]:
# split --
# context_dataset : 10.7K  train
# qa_dataset      : 10K train (drop 4.5K)

# merge them  then create three split

In [21]:
dataset = concatenate_datasets(
    [
        qa_dataset_preprocessed["train"],
        qa_dataset_preprocessed["test"],
        context_dataset_preprocessed1["train"],
        context_dataset_preprocessed2["train"],
        context_dataset_preprocessed2["test"],
    ]
).shuffle(seed=42)

In [22]:
dataset

Dataset({
    features: ['messages'],
    num_rows: 28704
})

In [23]:
dataset = dataset.train_test_split(test_size=0.25)
train_dataset, temp_dataset = dataset["train"], dataset["test"]

In [24]:
train_dataset, temp_dataset

(Dataset({
     features: ['messages'],
     num_rows: 21528
 }),
 Dataset({
     features: ['messages'],
     num_rows: 7176
 }))

In [25]:
temp_dataset = temp_dataset.train_test_split(test_size=0.50)
test_dataset, val_dataset = temp_dataset["train"], temp_dataset["test"]

In [26]:
test_dataset, val_dataset

(Dataset({
     features: ['messages'],
     num_rows: 3588
 }),
 Dataset({
     features: ['messages'],
     num_rows: 3588
 }))

In [27]:
print(torch.cuda.is_bf16_supported())  # true -> for even t4

True


In [28]:
test_dataset[100]

{'messages': [{'content': 'Answer the following question based on the given context \nQuestion - What is the total value of assets of the company? \nContext - And Advances: 0 OtherCurrentAssets: 61.64 Total Current Assets: 6,341.14 Total Assets: 6,536.53 OTHER ADDITIONAL INFORMATION: nan CONTINGENT LIABILITIES, COMMITMENTS: nan Contingent Liabilities: 60.96 CIF VALUE OF IMPORTS: nan Raw Materials: 0 Stores, Spares And Loose Tools: 0 Trade/Other Goods: 0 Capital Goods: 0 EXPENDITURE IN FOREIGN EXCHANGE: nan.\n',
   'role': 'user'},
  {'content': 'The total value of assets of the company is 6,536.53.',
   'role': 'assistant'}]}

In [31]:
print(
    tokenizer.decode(
        tokenizer.apply_chat_template(test_dataset[100]["messages"])["input_ids"],
        ignore_special_tokens=True,
    )
)

[BOS] Are ' no interest if paid in in x months ' credit cards worth it ? [CLS] [SEP] It has been reported in consumer media ( for example Clark Howard ' s radio program ) that the " no interest for 12 months " contracts could tr ick you with the terms and the dates on the contract . Just as an example : You borrow $ 1000 on 12 / 1 / 2013 , same as cash for 12 months . The contract will state the due date very clearly as 12 / 1 / 2014 . B UT they statements you get will take payment on the 15th of each month . So you will du ti fully pay your statements as they come in , but when you pay the final statement on 12 / 15 / 2014 , you are actually 14 days late , have violated the terms , and you now owe all the interest that accumulated ( and it was n ' t a favorable rate ). That does n ' t happen all the time . Not all contracts are written that way . But you better read your agreement . Some companies use the same as cash deal because they want to move product . Some do it because they wa

# Model demo before sft

In [38]:
from transformers import TextStreamer

streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

In [33]:
x = tokenizer(
    "The company reported a net revenue of $4.2 billion, representing a year-over-year growth of   ",
    return_tensors="pt",
).input_ids
x

tensor([[1308, 1797, 2853,   70, 1514, 1719, 1226,    9,   25,   19,   23, 2523,
           17, 4854,   70, 1421,   18, 1496,   18, 1421, 2018, 1226]])

In [42]:
model = model.to(device)

In [36]:
i = 0
for x in test_dataset:
    print("User - ", x["messages"][0]["content"], "\n\nAssistant - ", end=" ")
    inp = "[BOS] " + x["messages"][0]["content"] + "[SEP]"
    model.generate(
        tokenizer(inp, return_tensors="pt")["input_ids"].to(device),
        max_new_tokens=100,
        streamer=streamer,
    )
    print("\n\n")
    print("Actual answer - ", x["messages"][1]["content"])
    print("\n\n")
    i += 1
    if i == 10:
        break

User -  Need small buisness ideas with 100k $ budjet in a 3rd world country  

Assistant -  . The Company has developed a proprietary technology that is designed to be used in the production of high - quality , high - performance , high - performance , high - performance , high - performance , high - performance , high - performance , high - performance , high - performance , high - performance , high - performance , high - performance , high - performance , high - performance , high - performance , high - performance , high - performance , high - performance , high - performance , high - performance , high - 
 MAX TOKEN REACHED



User -  Pay off car or use money for down payment 

Assistant -  . We also offer a variety of other products and services to our customers , including : • “ car and car ” car sales , which are used to sell used car and car products , including car and car sales , which are used to sell used car and car products , including car and car sales , which are used 

# Pre sft evaluation

In [41]:
# # model.eval()
# from tqdm import tqdm
# candidates = []
# references = []
# with torch.no_grad():
#     for batch in tqdm(test_data_loader):
#         out = generate_prediction(batch,model)
#         candidates += out
#         references += batch["summaries"]
#         assert len(candidates) == len(references)

In [42]:
# import evaluate

# rouge = evaluate.load("rouge")
# bleu = evaluate.load("bleu")
# rg = rouge.compute(predictions=candidates, references=[[r] for r in references])
# bl = bleu.compute(predictions=candidates, references=references)
# rg, bl

# Sft trainer

In [43]:
!pip install trl -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 16.0 MB/s eta 0:00:0000:01


In [44]:
batch_size = 6
accumulation_steps = 4
epochs = 2

In [45]:
n_steps = (len(train_dataset) * epochs) / (batch_size * accumulation_steps * 2)
n_steps

897.0

In [46]:
val_dataset = val_dataset.select([i for i in range(500)])

In [47]:
x = tokenizer.apply_chat_template(
    val_dataset[0]["messages"], return_assistant_tokens_mask=True
)
print(tokenizer.decode(x["input_ids"]))
print(x["assistant_masks"])

[BOS] Answer the following question based on the given context Ques tion - What sections of the Annual Report on Form 10 - K contain the company ' s financial statements ? Cont ext - The consolidated financial statements and accompanying notes listed in Part IV , Item 15 ( a )( 1 ) of this Annual Report on Form 10 - K are included elsewhere in this Annual Report on Form 10 - K .. [CLS] [SEP] The consolidated financial statements are included in Part IV , Item 15 ( a )( 1 ). [EOS]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [48]:
print(tokenizer.chat_template)

{% for msg in messages %}{% if msg['role'] == 'user' %}[BOS] {{ msg['content'] }} [CLS]{% elif msg['role'] == 'assistant' %}{% generation %}[SEP] {{ msg['content'] }} [EOS]{% endgeneration %}{% endif %}{% endfor %}


In [49]:
val_dataset[0]["messages"]

[{'content': "Answer the following question based on the given context \nQuestion - What sections of the Annual Report on Form 10-K contain the company's financial statements? \nContext - The consolidated financial statements and accompanying notes listed in Part IV, Item 15(a)(1) of this Annual Report on Form 10-K are included elsewhere in this Annual Report on Form 10-K..\n",
  'role': 'user'},
 {'content': 'The consolidated financial statements are included in Part IV, Item 15(a)(1).',
  'role': 'assistant'}]

In [50]:
model.device

device(type='cuda', index=0)

In [51]:
# original_forward = model.forward

In [52]:
# import types

# cnt=0
# def debug_forward(self, input_ids=None, labels=None, **kwargs):
#     print("input_ids shape:", input_ids.shape)
#     print("labels shape:", labels.shape if labels is not None else None)
#     print("Input Ids: ",input_ids[0].tolist())
#     print("labels: ",labels[0].tolist())
#     input_ids=input_ids.to(self.model.embedding.weight.device)
#     if labels is not None:
#         labels=labels.to(self.model.embedding.weight.device)
#     x = original_forward(input_ids=input_ids, labels=labels, **kwargs)
#     global cnt
#     cnt+=1
#     print("loss:",x.loss.item())
#     if cnt==5:
#         return -1
#     return x

# # Bind it properly as a method
# model.forward = types.MethodType(debug_forward,model)

In [53]:
# model.forward=original_forward

In [54]:
from trl import SFTTrainer, SFTConfig

config = SFTConfig(
    output_dir="/kaggle/working/sft",
    save_strategy="steps",
    save_steps=50,
    per_device_train_batch_size=batch_size,
    num_train_epochs=epochs,
    logging_steps=10,
    gradient_checkpointing=False,
    bf16=True,
    fp16=False,
    gradient_accumulation_steps=accumulation_steps,
    batch_eval_metrics=True,
    eval_strategy="steps",
    eval_steps=50,
    save_total_limit=2,
    assistant_only_loss=True,
)

trainer = SFTTrainer(
    model=model,
    args=config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

The repository Pankaj121212/PM_MiniFinLLM contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/Pankaj121212/PM_MiniFinLLM .
 You can inspect the repository content at https://hf.co/Pankaj121212/PM_MiniFinLLM.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y
The repository Pankaj121212/PM_MiniFinLLM contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/Pankaj121212/PM_MiniFinLLM .
 You can inspect the repository content at https://hf.co/Pankaj121212/PM_MiniFinLLM.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


Tokenizing train dataset:   0%|          | 0/21528 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

In [2]:
!nvidia-smi

Sun May 24 14:37:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
trainer.train()

In [57]:
model.push_to_hub("Pankaj121212/sft_fin_llm")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/Pankaj121212/sft_fin_llm/commit/8a29f0bea33c75f40fd6e9b5aec47296c9920cb7', commit_message='Upload model', commit_description='', oid='8a29f0bea33c75f40fd6e9b5aec47296c9920cb7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Pankaj121212/sft_fin_llm', endpoint='https://huggingface.co', repo_type='model', repo_id='Pankaj121212/sft_fin_llm'), pr_revision=None, pr_num=None)

# Post sft Evaluation

In [1]:
# # model.eval()
# candidates = []
# references = []
# with torch.no_grad():
#     for batch in tqdm(test_data):
#         out = generate_prediction(batch,model)
#         candidates += out
#         references += batch["summaries"]
#         assert len(candidates) == len(references)

In [59]:
# import evaluate

# rouge = evaluate.load("rouge")
# bleu = evaluate.load("bleu")
# rg = rouge.compute(predictions=candidates, references=[[r] for r in references])
# bl = bleu.compute(predictions=candidates, references=references)
# rg, bl

In [4]:
model = AutoModelForCausalLM.from_pretrained(
    "Pankaj121212/sft_fin_llm", trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(
    "Pankaj121212/PM_MiniFinLLM", trust_remote_code=True
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

PM_MiniFinLLM_Model LOAD REPORT from: Pankaj121212/sft_fin_llm
Key                 | Status  | 
--------------------+---------+-
model.linear.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

configuration_PM_MiniFinLLM.py:   0%|          | 0.00/376 [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Pankaj121212/PM_MiniFinLLM:
- configuration_PM_MiniFinLLM.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.jinja:   0%|          | 0.00/214 [00:00<?, ?B/s]

In [34]:
model = model.to(device)
model.eval()

PM_MiniFinLLM_Model(
  (model): LLM(
    (embedding): Embedding(32000, 768)
    (embed_dropout): Dropout(p=0.1, inplace=False)
    (llm_layers): ModuleList(
      (0-11): 12 x Llm_layer(
        (linear1): Linear(in_features=768, out_features=2304, bias=True)
        (attn_layer): ScaledDotProductAttention(
          (softmax): Softmax(dim=-1)
        )
        (lyr_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (linear2): Linear(in_features=768, out_features=768, bias=True)
        (lyr_norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (ffn): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.1, inplace=False)
          (3): Linear(in_features=3072, out_features=768, bias=True)
        )
        (rope): RotaryPositionalEmbeddings()
        (linear_dropout): Dropout(p=0.1, inplace=False)
        (ffn_dropout): Dropout(p=0.1, inplace=False)
      )


In [1]:
i = 0
for x in test_dataset:
    print("User - ", x["messages"][0]["content"], "\n\nAssistant - ", end=" ")
    inp = "[BOS] " + x["messages"][0]["content"] + "[SEP]"
    # print(tokenizer(inp,return_tensors='pt'))
    model.generate(
        tokenizer(inp, return_tensors="pt")["input_ids"].to(device),
        max_new_tokens=100,
        streamer=streamer,
    )
    print("\n\n")
    print("Actual answer - ", x["messages"][1]["content"])
    print("\n\n")
    i += 1
    if i == 10:
        break